# 02 — Targets and Point-in-Time Feature Engineering

## Purpose

This notebook converts the quality-audited TT dataset into a **leakage-safe modelling dataset**.

It performs only five core tasks:

1. define the primary modelling calendar;
2. create direct H1, H3 and H7 forecast targets;
3. construct TT, cross-currency and macro/global features using only information available at the forecast origin;
4. lock the 2024–2025 final holdout;
5. create rolling 2019–2023 development folds with target-overlap protection.

## Forecasting convention

The forecast origin is the end of observation \(t\), after the current TT quotation is known.

Primary calendar: **weekdays only**.

Forecast horizons:
- H1 = 1 future weekday observation
- H3 = 3 future weekday observations
- H7 = 7 future weekday observations

The final 2024–2025 period is saved but remains **locked for model selection**.

In [ ]:
# ============================================================
# 1. IMPORTS + CONFIGURATION
# ============================================================

from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 150)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

CURRENCIES = ["USD", "GBP", "EUR", "JPY"]
HORIZONS = [1, 3, 7]

FINAL_HOLDOUT_START = pd.Timestamp("2024-01-01")
OUTER_VALIDATION_YEARS = [2019, 2020, 2021, 2022, 2023]

print("Configuration loaded.")

In [ ]:
# ============================================================
# 2. LOCATE PROJECT FILES
# ============================================================

def find_project_root():
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path.cwd().parent.parent,
    ]

    for root in candidates:
        if (root / "data" / "raw" / "Main set.xlsx").exists():
            return root.resolve()

    raise FileNotFoundError(
        "Research project root not found. "
        "Expected Research/data/raw/Main set.xlsx"
    )

PROJECT_ROOT = find_project_root()

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
TABLE_DIR = PROJECT_ROOT / "outputs" / "tables"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

AUDIT_PATH = PROCESSED_DIR / "01_quality_flagged_daily_dataset.csv"
RAW_PATH = RAW_DIR / "Main set.xlsx"

if AUDIT_PATH.exists():
    INPUT_PATH = AUDIT_PATH
    print("Using Notebook 01 audit dataset:", INPUT_PATH)
else:
    INPUT_PATH = RAW_PATH
    print("Notebook 01 audit CSV not found; using raw Excel:", INPUT_PATH)

In [ ]:
# ============================================================
# 3. LOAD DATA
# ============================================================

if INPUT_PATH.suffix.lower() == ".csv":
    daily = pd.read_csv(INPUT_PATH, parse_dates=["Date"])
else:
    daily = pd.read_excel(INPUT_PATH)
    daily["Date"] = pd.to_datetime(daily["Date"], errors="coerce")

daily = daily.sort_values("Date").reset_index(drop=True)

required = ["Date"]
for cur in CURRENCIES:
    required += [f"{cur}_buy", f"{cur}_sell", f"{cur}_mid"]

missing_required = [c for c in required if c not in daily.columns]
if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

print("Loaded:", daily.shape)
print("Date range:", daily["Date"].min().date(), "to", daily["Date"].max().date())

# 4. Primary modelling calendar

The raw dataset is daily, but weekend TT quotations are often carried forward unchanged.

For the primary experiment we use **Monday–Friday observations only**.

This does not claim every weekday is a true bank working day. A calendar-day sensitivity analysis can be performed later.

In [ ]:
# ============================================================
# 4. CREATE PRIMARY WEEKDAY CALENDAR
# ============================================================

daily["Day_of_Week"] = daily["Date"].dt.dayofweek
daily["Is_Weekend"] = daily["Day_of_Week"].isin([5, 6])

model_df = (
    daily.loc[~daily["Is_Weekend"]]
    .copy()
    .sort_values("Date")
    .reset_index(drop=True)
)

print("Daily rows   :", len(daily))
print("Weekday rows :", len(model_df))
print("Removed weekend rows:", len(daily) - len(model_df))
print("Weekday date range:", model_df["Date"].min().date(), "to", model_df["Date"].max().date())

# 5. Preserve TT values and create clean spread fields

The raw TT values are never overwritten.

If `buy > sell`, the buy/sell pair is treated as unreliable for spread-based features only.

The observed mid-rate is retained because it is the primary forecast series.

In [ ]:
# ============================================================
# 5. TT CLEAN FIELDS
# ============================================================

for cur in CURRENCIES:
    buy = pd.to_numeric(model_df[f"{cur}_buy"], errors="coerce")
    sell = pd.to_numeric(model_df[f"{cur}_sell"], errors="coerce")
    mid = pd.to_numeric(model_df[f"{cur}_mid"], errors="coerce")

    invalid_pair = buy > sell

    model_df[f"{cur}_buy_clean"] = buy.mask(invalid_pair)
    model_df[f"{cur}_sell_clean"] = sell.mask(invalid_pair)

    model_df[f"{cur}_spread"] = (
        model_df[f"{cur}_sell_clean"]
        - model_df[f"{cur}_buy_clean"]
    )

    model_df[f"{cur}_spread_pct"] = (
        model_df[f"{cur}_spread"]
        / mid.replace(0, np.nan)
    ) * 100

    model_df[f"{cur}_buy_gt_sell_flag"] = invalid_pair.astype(int)

cleaning_summary = pd.DataFrame({
    "Currency": CURRENCIES,
    "Buy_Greater_Than_Sell_Flags": [
        int(model_df[f"{c}_buy_gt_sell_flag"].sum())
        for c in CURRENCIES
    ]
})

display(cleaning_summary)
cleaning_summary.to_csv(
    TABLE_DIR / "02_tt_cleaning_summary.csv",
    index=False
)

# 6. Direct forecast targets

For each currency and horizon we save:

- future TT mid level;
- future absolute change;
- future percentage change;
- future log change;
- actual future target date.

These are **labels only** and must never be used as input features.

In [ ]:
# ============================================================
# 6. CREATE H1 / H3 / H7 TARGETS
# ============================================================

for h in HORIZONS:
    model_df[f"Target_Date_H{h}"] = model_df["Date"].shift(-h)

    for cur in CURRENCIES:
        current = model_df[f"{cur}_mid"]
        future = current.shift(-h)

        model_df[f"{cur}_target_H{h}"] = future
        model_df[f"{cur}_change_H{h}"] = future - current

        model_df[f"{cur}_pct_change_H{h}"] = (
            future / current.replace(0, np.nan) - 1
        )

        model_df[f"{cur}_log_change_H{h}"] = np.log(
            future / current.replace(0, np.nan)
        )

target_cols = [
    c for c in model_df.columns
    if (
        "_target_H" in c
        or "_change_H" in c
        or "_pct_change_H" in c
        or "_log_change_H" in c
        or c.startswith("Target_Date_H")
    )
]

print("Target columns created:", len(target_cols))
print(target_cols[:20])

In [ ]:
# ============================================================
# 7. TARGET SANITY CHECK
# ============================================================

target_checks = []

for h in HORIZONS:
    valid = model_df[f"Target_Date_H{h}"].notna()

    target_checks.append({
        "Horizon": h,
        "Valid_Target_Rows": int(valid.sum()),
        "Missing_Target_Rows_At_End": int((~valid).sum()),
        "Target_Date_After_Origin": bool(
            (
                model_df.loc[valid, f"Target_Date_H{h}"]
                > model_df.loc[valid, "Date"]
            ).all()
        )
    })

target_check_df = pd.DataFrame(target_checks)

display(target_check_df)

assert target_check_df["Target_Date_After_Origin"].all()

target_check_df.to_csv(
    TABLE_DIR / "02_target_sanity_check.csv",
    index=False
)

# 8. TT-derived point-in-time features

Current TT information is allowed because the forecast is made **after today's quotation is observed**.

Features include:
- current mid;
- clean buy/sell and spread;
- lagged TT levels;
- lagged returns;
- moving averages and EMAs;
- rolling ranges;
- rolling return volatility;
- momentum.

In [ ]:
# ============================================================
# 8. OWN-CURRENCY TT FEATURES
# ============================================================

LAGS = [1, 2, 3, 5, 7, 14, 30]
ROLL_WINDOWS = [3, 5, 7, 14, 30]
VOL_WINDOWS = [5, 7, 14, 30]

for cur in CURRENCIES:
    mid = model_df[f"{cur}_mid"]

    model_df[f"{cur}_log_return_1"] = np.log(
        mid / mid.shift(1)
    )
    model_df[f"{cur}_abs_log_return_1"] = (
        model_df[f"{cur}_log_return_1"].abs()
    )

    for lag in LAGS:
        model_df[f"{cur}_mid_lag{lag}"] = mid.shift(lag)
        model_df[f"{cur}_log_return_lag{lag}"] = (
            model_df[f"{cur}_log_return_1"].shift(lag)
        )

    for w in ROLL_WINDOWS:
        model_df[f"{cur}_mid_ma{w}"] = (
            mid.rolling(w, min_periods=w).mean()
        )

        model_df[f"{cur}_mid_ema{w}"] = (
            mid.ewm(span=w, adjust=False, min_periods=w).mean()
        )

        model_df[f"{cur}_mid_min{w}"] = (
            mid.rolling(w, min_periods=w).min()
        )

        model_df[f"{cur}_mid_max{w}"] = (
            mid.rolling(w, min_periods=w).max()
        )

        model_df[f"{cur}_mid_range{w}"] = (
            model_df[f"{cur}_mid_max{w}"]
            - model_df[f"{cur}_mid_min{w}"]
        )

    for w in VOL_WINDOWS:
        model_df[f"{cur}_return_std{w}"] = (
            model_df[f"{cur}_log_return_1"]
            .rolling(w, min_periods=w)
            .std()
        )

    for lag in [1, 3, 5, 7, 14, 30]:
        model_df[f"{cur}_momentum_{lag}"] = (
            mid / mid.shift(lag) - 1
        )

print("TT-derived features created.")

# 9. Cross-currency features

All four TT quotations are observed at the same forecast origin, so current cross-currency information is allowed.

We keep this compact:
- current foreign-currency TT mids;
- cross-currency lags 1, 3 and 7;
- current log returns;
- rolling return correlations.

In [ ]:
# ============================================================
# 9. CROSS-CURRENCY FEATURES
# ============================================================

for cur in CURRENCIES:
    for other in CURRENCIES:
        if other == cur:
            continue

        model_df[f"{cur}_x_{other}_mid_current"] = model_df[f"{other}_mid"]

        for lag in [1, 3, 7]:
            model_df[f"{cur}_x_{other}_mid_lag{lag}"] = (
                model_df[f"{other}_mid"].shift(lag)
            )

        model_df[f"{cur}_x_{other}_return_current"] = (
            model_df[f"{other}_log_return_1"]
        )

        for w in [14, 30]:
            model_df[f"{cur}_x_{other}_corr{w}"] = (
                model_df[f"{cur}_log_return_1"]
                .rolling(w, min_periods=w)
                .corr(model_df[f"{other}_log_return_1"])
            )

print("Cross-currency features created.")

# 10. Point-in-time macro/global features

The raw dataset contains variables with different update frequencies.

To avoid optimistic timing assumptions:

### Daily/global market variables
Use the previous weekday observation:
- DXY
- Crude oil
- VIX

### Policy rates
Use a one-observation lag:
- SDF
- SLF

### Periodic macroeconomic variables
Use the **previous observed value block**, not the current block.  
This is a conservative proxy when exact publication timestamps are unavailable.

### Annual GDP
Use the previous calendar year's value.

This prevents current-period macro totals from being treated as if they were known at the beginning of that period.

In [ ]:
# ============================================================
# 10. MACRO AVAILABILITY POLICY
# ============================================================

DAILY_GLOBAL = [
    "DXY",
    "Crude_Oil_Price",
    "VIXCLS",
]

POLICY_RATE_VARS = [
    "SDF_Rate",
    "SLF_Rate",
]

PERIODIC_MACRO = [
    "Remittances_USD_Mn",
    "Tourism_USD_Mn",
    "FX_Reserves_USD_Mn",
    "Direct_Tax",
    "Indirect_Tax",
    "Tax_Revenue_Total",
    "Non_Tax_Revenue",
    "Total_Revenue",
    "Total_Exports",
    "Total_Imports",
    "Political_Stability",
    "BOP",
    "CSE_Net_Inflow",
    "TBills_Inflow_Gross",
    "Inflation",
]

ANNUAL_MACRO = [
    "GDP_USD",
]

availability_rows = []

for c in DAILY_GLOBAL:
    if c in model_df.columns:
        availability_rows.append({
            "Variable": c,
            "Policy": "Lag 1 weekday observation",
            "Reason": "Conservative daily/global market availability"
        })

for c in POLICY_RATE_VARS:
    if c in model_df.columns:
        availability_rows.append({
            "Variable": c,
            "Policy": "Lag 1 weekday observation",
            "Reason": "Avoid same-origin timing ambiguity"
        })

for c in PERIODIC_MACRO:
    if c in model_df.columns:
        availability_rows.append({
            "Variable": c,
            "Policy": "Previous observed value block",
            "Reason": "Exact historical publication timestamp unavailable"
        })

for c in ANNUAL_MACRO:
    if c in model_df.columns:
        availability_rows.append({
            "Variable": c,
            "Policy": "Previous calendar-year value",
            "Reason": "Conservative annual availability"
        })

availability_policy = pd.DataFrame(availability_rows)

display(availability_policy)

availability_policy.to_csv(
    TABLE_DIR / "02_point_in_time_availability_policy.csv",
    index=False
)

In [ ]:
# ============================================================
# 11. HELPER: PREVIOUS CONSTANT-VALUE BLOCK
# ============================================================

def previous_value_block(series):
    s = pd.to_numeric(series, errors="coerce")

    # Each change starts a new observed-value block.
    block_id = s.ne(s.shift(1)).cumsum()

    # One value per block.
    block_value = s.groupby(block_id).first()

    # Previous block's value.
    previous_block_value = block_value.shift(1)

    # Map the previous block value back to every row
    # belonging to the current block.
    return block_id.map(previous_block_value)

print("Previous-block helper ready.")

In [ ]:
# ============================================================
# 12. CREATE PIT MACRO FEATURES
# ============================================================

PIT_MACRO_FEATURES = []

for c in DAILY_GLOBAL:
    if c in model_df.columns:
        name = f"{c}_PIT"
        model_df[name] = pd.to_numeric(
            model_df[c],
            errors="coerce"
        ).shift(1)
        PIT_MACRO_FEATURES.append(name)

for c in POLICY_RATE_VARS:
    if c in model_df.columns:
        name = f"{c}_PIT"
        model_df[name] = pd.to_numeric(
            model_df[c],
            errors="coerce"
        ).shift(1)
        PIT_MACRO_FEATURES.append(name)

for c in PERIODIC_MACRO:
    if c in model_df.columns:
        name = f"{c}_PIT"
        model_df[name] = previous_value_block(
            model_df[c]
        )
        PIT_MACRO_FEATURES.append(name)

for c in ANNUAL_MACRO:
    if c in model_df.columns:
        name = f"{c}_PIT"

        annual_lookup = (
            model_df
            .assign(_year=model_df["Date"].dt.year)
            .groupby("_year")[c]
            .last()
        )

        model_df[name] = (
            model_df["Date"].dt.year
            .map(annual_lookup.shift(1))
        )

        PIT_MACRO_FEATURES.append(name)

print("PIT macro/global features:", len(PIT_MACRO_FEATURES))
print(PIT_MACRO_FEATURES)

# 13. Calendar features

Only deterministic calendar information available at time \(t\) is included.

In [ ]:
# ============================================================
# 13. CALENDAR FEATURES
# ============================================================

model_df["cal_day_of_week"] = model_df["Date"].dt.dayofweek
model_df["cal_month"] = model_df["Date"].dt.month
model_df["cal_quarter"] = model_df["Date"].dt.quarter
model_df["cal_day_of_month"] = model_df["Date"].dt.day

model_df["cal_month_sin"] = np.sin(
    2 * np.pi * model_df["cal_month"] / 12
)
model_df["cal_month_cos"] = np.cos(
    2 * np.pi * model_df["cal_month"] / 12
)

CALENDAR_FEATURES = [
    "cal_day_of_week",
    "cal_month",
    "cal_quarter",
    "cal_day_of_month",
    "cal_month_sin",
    "cal_month_cos",
]

print("Calendar features created.")

# 14. Simple regime-state features

These are **not hindsight crisis labels**.

They are computed only from historical/current TT returns:
- recent volatility;
- recent absolute movement;
- recent trend.

This gives later models a simple description of the market state without using future information.

In [ ]:
# ============================================================
# 14. LEAKAGE-SAFE MARKET-STATE FEATURES
# ============================================================

for cur in CURRENCIES:
    r = model_df[f"{cur}_log_return_1"]

    model_df[f"{cur}_state_vol14"] = (
        r.rolling(14, min_periods=14).std()
    )

    model_df[f"{cur}_state_absret14"] = (
        r.abs().rolling(14, min_periods=14).mean()
    )

    model_df[f"{cur}_state_trend14"] = (
        model_df[f"{cur}_mid"]
        / model_df[f"{cur}_mid"].shift(14)
        - 1
    )

print("Simple leakage-safe state features created.")

# 15. Feature families

To keep modelling interpretable, later notebooks will compare only four feature families:

- `TT_OWN`
- `TT_CROSS`
- `TT_MACRO`
- `FULL_PIT`

Target columns are explicitly excluded.

In [ ]:
# ============================================================
# 15. BUILD FEATURE SETS
# ============================================================

TARGET_MARKERS = [
    "_target_H",
    "_change_H",
    "_pct_change_H",
    "_log_change_H",
]

def is_target_column(col):
    if col.startswith("Target_Date_H"):
        return True
    return any(marker in col for marker in TARGET_MARKERS)

def safe_feature_list(cols):
    out = []

    for c in cols:
        if c not in model_df.columns:
            continue
        if c == "Date":
            continue
        if is_target_column(c):
            continue
        if c not in out:
            out.append(c)

    return out

FEATURE_SETS = {}

for cur in CURRENCIES:
    own = []

    # Current clean TT information.
    own += [
        f"{cur}_mid",
        f"{cur}_buy_clean",
        f"{cur}_sell_clean",
        f"{cur}_spread",
        f"{cur}_spread_pct",
        f"{cur}_log_return_1",
        f"{cur}_abs_log_return_1",
    ]

    # Own-currency generated features.
    own += [
        c for c in model_df.columns
        if (
            c.startswith(f"{cur}_mid_lag")
            or c.startswith(f"{cur}_log_return_lag")
            or c.startswith(f"{cur}_mid_ma")
            or c.startswith(f"{cur}_mid_ema")
            or c.startswith(f"{cur}_mid_min")
            or c.startswith(f"{cur}_mid_max")
            or c.startswith(f"{cur}_mid_range")
            or c.startswith(f"{cur}_return_std")
            or c.startswith(f"{cur}_momentum_")
            or c.startswith(f"{cur}_state_")
        )
    ]

    own += CALENDAR_FEATURES
    own = safe_feature_list(own)

    cross = own + [
        c for c in model_df.columns
        if c.startswith(f"{cur}_x_")
    ]
    cross = safe_feature_list(cross)

    macro = own + PIT_MACRO_FEATURES
    macro = safe_feature_list(macro)

    full = cross + PIT_MACRO_FEATURES
    full = safe_feature_list(full)

    FEATURE_SETS[cur] = {
        "TT_OWN": own,
        "TT_CROSS": cross,
        "TT_MACRO": macro,
        "FULL_PIT": full,
    }

feature_count_rows = []

for cur in CURRENCIES:
    for family, features in FEATURE_SETS[cur].items():
        feature_count_rows.append({
            "Currency": cur,
            "Feature_Family": family,
            "Feature_Count": len(features)
        })

feature_count_df = pd.DataFrame(feature_count_rows)

display(feature_count_df)

In [ ]:
# ============================================================
# 16. HARD LEAKAGE AUDIT OF FEATURE NAMES
# ============================================================

leakage_rows = []

for cur in CURRENCIES:
    for family, features in FEATURE_SETS[cur].items():
        forbidden = [
            c for c in features
            if is_target_column(c)
        ]

        leakage_rows.append({
            "Currency": cur,
            "Feature_Family": family,
            "Feature_Count": len(features),
            "Forbidden_Target_Features": "|".join(forbidden),
            "Leakage_Name_Check": "PASS" if not forbidden else "FAIL"
        })

leakage_audit = pd.DataFrame(leakage_rows)

display(leakage_audit)

if not leakage_audit["Leakage_Name_Check"].eq("PASS").all():
    raise RuntimeError("Target leakage detected in feature sets.")

leakage_audit.to_csv(
    TABLE_DIR / "02_feature_leakage_audit.csv",
    index=False
)

# 17. Lock development and final holdout

The final holdout starts on:

\[
2024-01-01
\]

No later model-development notebook should use the holdout for:
- feature selection;
- hyperparameter tuning;
- model selection;
- threshold selection.

Only the final holdout notebook may evaluate it.

In [ ]:
# ============================================================
# 17. SPLIT FLAG
# ============================================================

model_df["Research_Split"] = np.where(
    model_df["Date"] < FINAL_HOLDOUT_START,
    "Development",
    "Final_Holdout_LOCKED"
)

split_summary = (
    model_df.groupby("Research_Split")
    .agg(
        Rows=("Date", "size"),
        Start_Date=("Date", "min"),
        End_Date=("Date", "max"),
    )
    .reset_index()
)

display(split_summary)

development = (
    model_df[
        model_df["Research_Split"] == "Development"
    ]
    .copy()
    .reset_index(drop=True)
)

holdout_locked = (
    model_df[
        model_df["Research_Split"] == "Final_Holdout_LOCKED"
    ]
    .copy()
    .reset_index(drop=True)
)

assert development["Date"].max() < FINAL_HOLDOUT_START

print("Development rows:", len(development))
print("Locked holdout rows:", len(holdout_locked))

# 18. Rolling development folds

Outer validation years are:

2019, 2020, 2021, 2022, 2023.

For each horizon, a training observation is accepted only if its **target date is strictly before the validation start date**.

This is stronger and clearer than relying only on row-count embargo logic.

In [ ]:
# ============================================================
# 18. CREATE OUTER FOLD DEFINITIONS
# ============================================================

fold_rows = []

for year in OUTER_VALIDATION_YEARS:
    val_mask = development["Date"].dt.year.eq(year)

    if not val_mask.any():
        continue

    val_start = development.loc[val_mask, "Date"].min()
    val_end = development.loc[val_mask, "Date"].max()

    for h in HORIZONS:
        target_date_col = f"Target_Date_H{h}"

        supervised_train_mask = (
            (development["Date"] < val_start)
            & development[target_date_col].notna()
            & (development[target_date_col] < val_start)
        )

        val_h_mask = (
            val_mask
            & development[target_date_col].notna()
        )

        train_dates = development.loc[
            supervised_train_mask,
            "Date"
        ]

        fold_rows.append({
            "Validation_Year": year,
            "Horizon": h,
            "Train_Start": train_dates.min(),
            "Train_End": train_dates.max(),
            "Train_Rows": int(supervised_train_mask.sum()),
            "Validation_Start": val_start,
            "Validation_End": val_end,
            "Validation_Rows": int(val_h_mask.sum()),
            "Max_Training_Target_Date": development.loc[
                supervised_train_mask,
                target_date_col
            ].max(),
        })

outer_folds = pd.DataFrame(fold_rows)

display(outer_folds)

outer_folds.to_csv(
    TABLE_DIR / "02_outer_rolling_folds.csv",
    index=False
)

In [ ]:
# ============================================================
# 19. TARGET-OVERLAP AUDIT
# ============================================================

overlap_rows = []

for _, row in outer_folds.iterrows():
    max_target = pd.to_datetime(
        row["Max_Training_Target_Date"]
    )
    val_start = pd.to_datetime(
        row["Validation_Start"]
    )

    passed = (
        pd.notna(max_target)
        and max_target < val_start
    )

    overlap_rows.append({
        "Validation_Year": int(row["Validation_Year"]),
        "Horizon": int(row["Horizon"]),
        "Max_Training_Target_Date": max_target,
        "Validation_Start": val_start,
        "Target_Overlap_Audit": "PASS" if passed else "FAIL"
    })

target_overlap_audit = pd.DataFrame(overlap_rows)

display(target_overlap_audit)

if not target_overlap_audit["Target_Overlap_Audit"].eq("PASS").all():
    raise RuntimeError(
        "Target overlap detected between training labels and validation."
    )

target_overlap_audit.to_csv(
    TABLE_DIR / "02_target_overlap_audit.csv",
    index=False
)

# 20. Feature completeness

This does not remove incomplete rows.

It simply reports how much of each feature family is available after lagging and point-in-time transformations.

In [ ]:
# ============================================================
# 20. FEATURE COMPLETENESS
# ============================================================

completeness_rows = []

for cur in CURRENCIES:
    for family, features in FEATURE_SETS[cur].items():
        if not features:
            continue

        cell_completeness = (
            development[features]
            .notna()
            .to_numpy()
            .mean()
            * 100
        )

        complete_rows = (
            development[features]
            .notna()
            .all(axis=1)
            .mean()
            * 100
        )

        completeness_rows.append({
            "Currency": cur,
            "Feature_Family": family,
            "Feature_Count": len(features),
            "Cell_Completeness_Percent": cell_completeness,
            "Completely_Filled_Rows_Percent": complete_rows,
        })

feature_completeness = pd.DataFrame(completeness_rows)

display(feature_completeness)

feature_completeness.to_csv(
    TABLE_DIR / "02_feature_completeness.csv",
    index=False
)

# 21. Feature manifest

The manifest makes the engineered dataset auditable and easier to explain in the thesis/viva.

In [ ]:
# ============================================================
# 21. CREATE FEATURE MANIFEST
# ============================================================

manifest_rows = []

for cur in CURRENCIES:
    for family, features in FEATURE_SETS[cur].items():
        for feature in features:

            if feature in PIT_MACRO_FEATURES:
                source_group = "PIT Macro/Global"

            elif feature in CALENDAR_FEATURES:
                source_group = "Calendar"

            elif feature.startswith(f"{cur}_x_"):
                source_group = "Cross-Currency"

            else:
                source_group = "Own TT"

            manifest_rows.append({
                "Currency": cur,
                "Feature_Family": family,
                "Feature": feature,
                "Source_Group": source_group,
            })

feature_manifest = (
    pd.DataFrame(manifest_rows)
    .drop_duplicates()
    .sort_values(
        ["Currency", "Feature_Family", "Feature"]
    )
    .reset_index(drop=True)
)

display(feature_manifest.head(50))

feature_manifest.to_csv(
    TABLE_DIR / "02_feature_manifest.csv",
    index=False
)

# 22. Save modelling artifacts

The main file for later development notebooks is:

`02_development_model_dataset.csv`

The holdout is saved separately and clearly marked as locked.

Do not load `02_final_holdout_LOCKED.csv` in model-selection notebooks.

In [ ]:
# ============================================================
# 22. SAVE DATASETS + FEATURE DEFINITIONS
# ============================================================

FULL_PATH = (
    PROCESSED_DIR /
    "02_full_point_in_time_model_dataset.csv"
)

DEV_PATH = (
    PROCESSED_DIR /
    "02_development_model_dataset.csv"
)

HOLDOUT_PATH = (
    PROCESSED_DIR /
    "02_final_holdout_LOCKED.csv"
)

FEATURE_SET_JSON = (
    PROCESSED_DIR /
    "02_feature_sets.json"
)

model_df.to_csv(
    FULL_PATH,
    index=False
)

development.to_csv(
    DEV_PATH,
    index=False
)

holdout_locked.to_csv(
    HOLDOUT_PATH,
    index=False
)

with open(
    FEATURE_SET_JSON,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        FEATURE_SETS,
        f,
        indent=2
    )

print("Saved:")
print(" ", FULL_PATH)
print(" ", DEV_PATH)
print(" ", HOLDOUT_PATH)
print(" ", FEATURE_SET_JSON)

In [ ]:
# ============================================================
# 23. FINAL QUALITY GATE
# ============================================================

quality_gate = pd.DataFrame([
    {
        "Area": "Primary calendar",
        "Status": "PASS",
        "Evidence": "Monday-Friday observations only"
    },
    {
        "Area": "Forecast targets",
        "Status": "PASS",
        "Evidence": "Direct H1/H3/H7 future weekday-observation targets"
    },
    {
        "Area": "TT anomalies",
        "Status": "PASS",
        "Evidence": "Raw TT retained; invalid buy/sell pairs excluded only from spread features"
    },
    {
        "Area": "Daily/global PIT handling",
        "Status": "PASS",
        "Evidence": "DXY, oil and VIX use lag-1 weekday values"
    },
    {
        "Area": "Policy-rate PIT handling",
        "Status": "PASS",
        "Evidence": "SDF/SLF use lag-1 weekday values"
    },
    {
        "Area": "Periodic macro PIT handling",
        "Status": "PASS",
        "Evidence": "Previous observed value block used"
    },
    {
        "Area": "Annual macro PIT handling",
        "Status": "PASS",
        "Evidence": "Previous calendar-year GDP value used"
    },
    {
        "Area": "Feature leakage",
        "Status": (
            "PASS"
            if leakage_audit["Leakage_Name_Check"].eq("PASS").all()
            else "FAIL"
        ),
        "Evidence": "No target columns included in feature sets"
    },
    {
        "Area": "Target overlap",
        "Status": (
            "PASS"
            if target_overlap_audit["Target_Overlap_Audit"].eq("PASS").all()
            else "FAIL"
        ),
        "Evidence": "Every training target date precedes its validation start"
    },
    {
        "Area": "Final holdout",
        "Status": "PASS",
        "Evidence": "2024-2025 stored separately as LOCKED"
    },
])

display(quality_gate)

quality_gate.to_csv(
    TABLE_DIR / "02_quality_gate.csv",
    index=False
)

In [ ]:
# ============================================================
# 24. FINAL SUMMARY
# ============================================================

summary = {
    "weekday_rows_total": int(len(model_df)),
    "development_rows": int(len(development)),
    "locked_holdout_rows": int(len(holdout_locked)),
    "development_start": str(development["Date"].min().date()),
    "development_end": str(development["Date"].max().date()),
    "holdout_start": str(holdout_locked["Date"].min().date()),
    "holdout_end": str(holdout_locked["Date"].max().date()),
    "horizons": HORIZONS,
    "currencies": CURRENCIES,
    "feature_set_counts": {
        cur: {
            family: len(features)
            for family, features in FEATURE_SETS[cur].items()
        }
        for cur in CURRENCIES
    },
}

with open(
    TABLE_DIR / "02_summary.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        summary,
        f,
        indent=2
    )

print("=" * 76)
print("NOTEBOOK 02 — TARGETS + POINT-IN-TIME FEATURES COMPLETE")
print("=" * 76)

print("Primary weekday rows :", len(model_df))
print("Development rows     :", len(development))
print("Locked holdout rows  :", len(holdout_locked))

print(
    "Development range    :",
    development["Date"].min().date(),
    "to",
    development["Date"].max().date()
)

print(
    "Locked holdout range :",
    holdout_locked["Date"].min().date(),
    "to",
    holdout_locked["Date"].max().date()
)

print("\nFeature family sizes:")
display(feature_count_df)

print("\nTarget-overlap audit:")
display(target_overlap_audit)

print("\nIMPORTANT:")
print("Do not use 02_final_holdout_LOCKED.csv for model development.")
print("Next notebook: 03_EDA_Diagnostics_and_Regimes.ipynb")